# 02. 경량 모델 파인튜닝

**목표**

- 작은 `TinySensorNet`을 기준 장치 데이터로 사전학습한다.
- 새 장치의 적은 데이터에 맞게 분류 head를 교체한다.
- backbone을 얼린 head-only 학습 후 작은 learning rate로 전체를 푼다.
- 모델 가중치와 C++ 전처리용 metadata를 함께 저장한다.

**사전 준비**: `python -m pip install -r requirements.txt`

전체 구현은 `src/fine_tune.py`에 있으며 모든 실행문 바로 위에 문법과 작성 이유가 한국어 주석으로 적혀 있다.

## 1. 모델 구조를 확인한다

| 줄 | 문법 | 이유 |
|---|---|---|
| 1 | `from src... import` | 학습 script와 노트북이 같은 모델 정의를 공유한다. |
| 2 | constructor | parameter가 등록된 `nn.Module` 객체를 만든다. |
| 3 | `sum(generator)` | 각 parameter tensor의 원소 수를 합쳐 모델 규모를 센다. |
| 4 | 출력 | layer와 총 parameter 수를 확인한다. |

In [ ]:
# fine_tune.py와 똑같은 TinySensorNet class를 가져온다.
from src.fine_tune import TinySensorNet
# 정상·주의·정지 필요의 세 logits를 출력하는 모델을 만든다.
model = TinySensorNet(num_classes=3)
# numel은 각 parameter tensor의 scalar 개수를 반환하며 sum이 전체를 합친다.
parameter_count = sum(parameter.numel() for parameter in model.parameters())
# 구조와 크기를 함께 보아 정말 경량인지 확인한다.
print(model, 'parameters=', parameter_count)

## 2. 전체 파인튜닝 workflow를 실행한다

아래 `%run`은 notebook magic으로 Python 파일을 현재 kernel에서 실행한다. 첫 실행은 학습 결과를 보기 쉽게 epoch를 짧게 설정했다. 정확한 비교 실험에서는 epoch를 늘리고 validation early stopping을 추가한다.

| 인수 | 뜻 |
|---|---|
| `--pretrain-epochs 30` | 기준 장치에서 backbone과 head를 함께 학습하는 횟수 |
| `--head-epochs 12` | backbone을 얼리고 새 head만 학습하는 횟수 |
| `--unfreeze-epochs 12` | 전체 모델을 작은 learning rate로 미세조정하는 횟수 |
| `--artifact-dir artifacts` | checkpoint와 metadata를 저장할 위치 |

In [ ]:
# %run은 script의 argparse에도 아래 문자열을 명령행 인수처럼 전달한다.
%run src/fine_tune.py --pretrain-epochs 30 --head-epochs 12 --unfreeze-epochs 12 --artifact-dir artifacts

## 3. metadata 계약을 검사한다

| 줄 | 문법 | 이유 |
|---|---|---|
| 1 | 표준 라이브러리 import | JSON 텍스트를 Python dictionary로 바꾼다. |
| 2 | `Path` import | 문자열 결합보다 안전한 경로 연산을 한다. |
| 3 | `with ... open` | 파일을 읽고 cell이 끝나기 전에 자동으로 닫는다. |
| 4 | `json.load` | file object에서 JSON 한 개를 읽는다. |
| 5 | dictionary 출력 | C++이 재현할 shape·dtype·mean·std·class 순서를 검토한다. |

In [ ]:
# json은 사람이 읽을 수 있고 여러 언어에서 지원하는 metadata 형식이다.
import json
# Path는 `/` 연산자로 폴더와 파일 이름을 결합할 수 있다.
from pathlib import Path
# UTF-8 읽기 모드로 metadata 파일을 열고 scope가 끝나면 닫는다.
with (Path('artifacts') / 'metadata.json').open('r', encoding='utf-8') as file:
    # load는 열린 파일의 JSON object를 Python dictionary로 변환한다.
    metadata = json.load(file)
# 입력 계약과 평가 점수를 notebook에서 확인한다.
print(json.dumps(metadata, ensure_ascii=False, indent=2))

## 파인튜닝 단계가 필요한 이유

1. **head 교체**: 원래 작업의 클래스 의미를 새 작업에 맞춘다.
2. **backbone freeze**: 적은 새 데이터가 이미 배운 일반 특징을 즉시 망가뜨리지 않게 한다.
3. **head-only 학습**: 새 label에 맞는 결정 경계를 빠르게 찾는다.
4. **작은 learning rate로 unfreeze**: 새 장치 분포에 특징도 조금 적응시킨다.
5. **독립 test 평가**: 학습 중 본 validation을 반복해서 선택한 데 따른 낙관을 줄인다.

이미지의 MobileNetV3 같은 사전학습 모델도 원리는 같다. 단, 원래 모델의 resize, channel order, mean/std를 그대로 시작점으로 사용하고 새 데이터 라이선스와 클래스 불균형을 확인해야 한다.

## 직접 해 볼 과제

1. `shift=0.20`으로 바꾸고 head-only와 unfreeze 뒤 정확도를 각각 기록한다.
2. backbone을 처음부터 모두 푼 결과와 비교한다.
3. target 학습 샘플을 클래스당 5개로 줄여 과적합을 관찰한다.
4. 위험 클래스 `stop_required`의 recall을 계산해 전체 정확도와 비교한다.
5. 가장 작은 정확도 하락으로 parameter 수를 줄일 수 있도록 hidden 차원 16, 8을 절반으로 바꾼다.

**통과 기준**: `artifacts/sensor_model.pt`, `artifacts/metadata.json` 생성, metadata의 input shape가 `[1, 4]`, 클래스 순서가 코드와 일치.